Transformer Prerequisites
====

Before building the full Transformer encoder-decoder architecture, we first need a few reusable input-processing components.

- The original Transformer used a subword tokenizer based on Byte Pair Encoding (BPE).
- It used a shared source-target vocabulary with about 37,000 tokens for the English-German translation setup.
- This is different from GPT-2's byte-level BPE tokenizer, which has no dedicated padding token.
- For this educational implementation, we will train a small classic BPE tokenizer with explicit special tokens:
  - `[PAD]` for padding batches to the same length
  - `[BOS]` for the beginning of a sequence
  - `[EOS]` for the end of a sequence
  - `[UNK]` for unknown tokens
- We will build padding masks from sequence lengths, instead of assuming a token ID always means padding.
- We will implement sinusoidal positional embeddings, matching the original Transformer instead of learned positional embeddings.
- Finally, we will combine token embeddings and positional embeddings to produce the input tensor used by the encoder and decoder blocks.


### 1. Classic BPE tokenizer

The original Transformer used BPE subword tokenization with a shared source-target vocabulary for translation. In this notebook, we use Hugging Face `tokenizers` to train a small shared BPE tokenizer on bilingual English-German text.

For a real English-German model, train this tokenizer on the combined source and target text from the training corpus. The tiny examples below are only for illustrating the mechanics.

In [1]:
from pathlib import Path

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing
from tokenizers.trainers import BpeTrainer

In [ ]:
def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing pyproject.toml")


repo_root = find_repo_root()
tokenizer_dir = repo_root / "data" / "tokenizers"
tokenizer_dir.mkdir(parents=True, exist_ok=True)

tokenizer_path = tokenizer_dir / "classic_bpe_en_de.json"
print(f"Tokenizer path: {tokenizer_path}")

In [3]:
english_german_examples = [
    "Hello, how are you?",
    "Hallo, wie geht es dir?",
    "The Transformer architecture uses attention mechanisms.",
    "Die Transformer-Architektur verwendet Attention-Mechanismen.",
    "The encoder reads the source sentence.",
    "Der Encoder liest den Quellsatz.",
    "The decoder generates the target sentence.",
    "Der Decoder erzeugt den Zielsatz.",
    "Machine translation maps text from one language to another.",
    "Maschinelle Übersetzung bildet Text von einer Sprache in eine andere ab.",
]

# Original-Transformer scale was about 37,000 tokens.
# This tiny demo corpus is too small for that, so we use a smaller vocabulary.
vocab_size = 1_000
special_tokens = ["[PAD]", "[BOS]", "[EOS]", "[UNK]"]

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=vocab_size,
    special_tokens=special_tokens,
    show_progress=False,
)

tokenizer.train_from_iterator(english_german_examples, trainer=trainer)

tokenizer.post_processor = TemplateProcessing(
    single="[BOS] $A [EOS]",
    pair="[BOS] $A [EOS] $B:1 [EOS]:1",
    special_tokens=[
        ("[BOS]", tokenizer.token_to_id("[BOS]")),
        ("[EOS]", tokenizer.token_to_id("[EOS]")),
    ],
)

tokenizer.save(str(tokenizer_path))

print(f"Trained vocabulary size: {tokenizer.get_vocab_size()}")
for token in special_tokens:
    print(f"{token}: {tokenizer.token_to_id(token)}")

Trained vocabulary size: 221
[PAD]: 0
[BOS]: 1
[EOS]: 2
[UNK]: 3


In [4]:
# Load the tokenizer from disk, like we would in later notebooks.
tokenizer = Tokenizer.from_file(str(tokenizer_path))

text_en = "The encoder uses attention."
text_de = "Der Encoder verwendet Attention."

encoded_en = tokenizer.encode(text_en)
encoded_de = tokenizer.encode(text_de)

print("English tokens:", encoded_en.tokens)
print("English ids:", encoded_en.ids)
print("German tokens:", encoded_de.tokens)
print("German ids:", encoded_de.ids)

English tokens: ['[BOS]', 'The', 'encoder', 'uses', 'attention', '.', '[EOS]']
English ids: [1, 57, 165, 154, 168, 6, 2]
German tokens: ['[BOS]', 'Der', 'Encoder', 'verwendet', 'Attention', '.', '[EOS]']
German ids: [1, 66, 184, 220, 183, 6, 2]


This tokenizer works for bilingual English-German text because BPE learns subword pieces from whatever training text it sees. If we train one shared tokenizer on both English source sentences and German target sentences, both languages use the same vocabulary and embedding table.

For real experiments, replace the small `english_german_examples` list with lines from a proper parallel corpus, such as WMT English-German data, and increase `vocab_size` toward the original scale.